# Bronze Ingestion — TfL Stop Points

Ingest London Underground stop-point reference data from the TfL Unified API.

This notebook:

1. Loads source configuration.
2. Calls the TfL StopPoint API.
3. Validates the response.
4. Lands the original JSON in the Unity Catalog Volume.
5. Appends the response to the Bronze Delta table.

**Source:** TfL Unified API  
**Target:** `workspace.urbanpulse_bronze.tfl_stop_points`

## 1. Initialise project paths

Add the repository `src` directory to the Python path so shared UrbanPulse modules can be imported.

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Project root: {PROJECT_ROOT}")

## 2. Import reusable ingestion components

In [0]:
import uuid

from urbanpulse.ingestion.api_clients import ApiClient
from urbanpulse.ingestion.bronze import write_raw_bronze
from urbanpulse.ingestion.landing import land_json
from urbanpulse.utils.config import load_yaml

## 3. Load source configuration

In [0]:
CONFIG_PATH = PROJECT_ROOT / "conf" / "sources.yml"

config = load_yaml(str(CONFIG_PATH))

tfl_config = config["tfl"]
stop_points_config = tfl_config["stop_points"]

BASE_URL = tfl_config["base_url"]
ENDPOINT = stop_points_config["endpoint"]

LANDING_PATH = (
    "/Volumes/workspace/"
    "urbanpulse_meta/"
    "landing"
)

BRONZE_TABLE = (
    "workspace."
    "urbanpulse_bronze."
    "tfl_stop_points"
)

print(f"Source: {BASE_URL}{ENDPOINT}")
print(f"Target: {BRONZE_TABLE}")

## 4. Request TfL Tube stop points

The StopPoint endpoint returns station/reference data for the requested transport mode.

In [0]:
client = ApiClient(
    base_url=BASE_URL
)

payload, status_code = client.get(
    endpoint=ENDPOINT
)

request_id = str(uuid.uuid4())

print(f"HTTP status: {status_code}")
print(f"Response type: {type(payload)}")
print(f"Request ID: {request_id}")

## 5. Validate the API response

The response must contain a non-empty `stopPoints` collection before it is persisted.

In [0]:
if status_code != 200:
    raise RuntimeError(
        f"TfL returned HTTP {status_code}"
    )

if not isinstance(payload, dict):
    raise TypeError(
        "Expected TfL StopPoint response to be a dictionary"
    )

if "stopPoints" not in payload:
    raise ValueError(
        "TfL response does not contain 'stopPoints'"
    )

stop_points = payload["stopPoints"]

if not isinstance(stop_points, list):
    raise TypeError(
        "'stopPoints' must be a list"
    )

if not stop_points:
    raise ValueError(
        "TfL returned no Tube stop points"
    )

print(
    f"Validation passed: "
    f"{len(stop_points)} stop points returned"
)

## 6. Inspect one stop point

Inspect one source record before persisting the response.

In [0]:
stop_points[0]

## 7. Land the original JSON response

Retain the complete source response for replay and debugging.

In [0]:
landing_file = land_json(
    payload=payload,
    base_path=LANDING_PATH,
    source="tfl",
    dataset="stop_points",
    request_id=request_id,
)

print(f"Raw file landed: {landing_file}")

## 8. Append to Bronze

Persist the complete TfL response and ingestion metadata as a Delta snapshot.

In [0]:
write_raw_bronze(
    spark=spark,
    payload=payload,
    request_id=request_id,
    source="tfl",
    dataset="stop_points",
    source_endpoint=ENDPOINT,
    http_status=status_code,
    table_name=BRONZE_TABLE,
)

print(
    f"Bronze ingestion completed: "
    f"{request_id}"
)

## 9. Verify the landing file

In [0]:
landing_parent = str(
    Path(landing_file).parent
)

display(
    dbutils.fs.ls(
        landing_parent
    )
)

## 10. Verify Bronze

In [0]:
%sql
SELECT
    request_id,
    source,
    dataset,
    source_endpoint,
    ingested_at,
    http_status,
    LENGTH(payload) AS payload_size
FROM workspace.urbanpulse_bronze.tfl_stop_points
ORDER BY ingested_at DESC;

## 11. Verify snapshot history

In [0]:
%sql
SELECT COUNT(*) AS total_snapshots
FROM workspace.urbanpulse_bronze.tfl_stop_points;